# Actividad 2  Perfilado y diagnóstico de calidad de datos

**Variante 5:** Rendimiento comercial por tienda y por empleado.

## Punto 2.1 – Reconocimiento del conjunto de datos

**Objetivo:** explorar la estructura, los tipos de datos y las características generales del conjunto de trabajo utilizado en la Actividad 1.

Se analizarán la cantidad de filas y columnas, los tipos de datos, las primeras filas, una muestra aleatoria, la información general y las estadísticas descriptivas con percentiles adicionales.

In [15]:

import sys
from pathlib import Path

carpeta_actual = Path.cwd().resolve()
carpeta_principal = (
    carpeta_actual.parent
    if carpeta_actual.name == "notebooks"
    else carpeta_actual
)

sys.path.insert(0, str(carpeta_principal))
os.chdir(carpeta_principal)

In [16]:
# ACTIVIDAD 2  CONEXIÓN A SQL SERVER
# Objetivo: conectarnos a Sakila_Espanol utilizando
# las credenciales guardadas en el archivo .env.

import os
import pandas as pd
import mssql_python
import certifi
from dotenv import load_dotenv
from esquema import nombre_tabla_sql, nombre_columna_sql

# Cargamos las variables de entorno.
load_dotenv()

# Configuramos la ubicación de los certificados SSL.
os.environ["SSL_CERT_FILE"] = certifi.where()

# Obtenemos los datos de conexión del archivo .env.
servidor = os.getenv("DB_SERVER")
puerto = os.getenv("DB_PORT")
base_datos = os.getenv("DB_NAME")
usuario = os.getenv("DB_USER")
contrasena = os.getenv("DB_PASSWORD")

# Establecemos la conexión con SQL Server.
# Utilizamos el mismo método que funcionó en la Actividad 1.
conexion = mssql_python.connect(
    f"SERVER={servidor},{puerto};"
    f"DATABASE={base_datos};"
    f"UID={usuario};"
    f"PWD={contrasena};"
    "Encrypt=yes;"
    "TrustServerCertificate=yes;"
)

# Verificamos a qué base de datos estamos conectados.
cursor = conexion.cursor()
cursor.execute("SELECT DB_NAME()")

resultado = cursor.fetchone()

print("¡Conexión exitosa!")
print("Base de datos:", resultado[0])

cursor.close()

# Mantenemos la conexión abierta para las siguientes consultas.

¡Conexión exitosa!
Base de datos: Sakila_Espanol


In [17]:
# ACTIVIDAD 2  PUNTO 2.1
# Cargamos el conjunto de trabajo para analizar su calidad.

# Recuperamos los pagos y los relacionamos con los empleados
# para identificar la tienda correspondiente a cada operación.

consulta_perfilado = f"""
SELECT
    p.{nombre_columna_sql('payment', 'payment_id')} AS pago_id,
    p.{nombre_columna_sql('payment', 'customer_id')} AS cliente_id,
    p.{nombre_columna_sql('payment', 'staff_id')} AS empleado_id,
    e.{nombre_columna_sql('staff', 'store_id')} AS tienda_id,
    p.{nombre_columna_sql('payment', 'amount')} AS monto,
    p.{nombre_columna_sql('payment', 'payment_date')} AS fecha_pago
FROM {nombre_tabla_sql('payment')} AS p
LEFT JOIN {nombre_tabla_sql('staff')} AS e
    ON p.{nombre_columna_sql('payment', 'staff_id')} = e.{nombre_columna_sql('staff', 'staff_id')}
"""

# Cargamos los resultados en un DataFrame de pandas.
df_perfilado = pd.read_sql(consulta_perfilado, conexion)

# Comprobamos que se cargaron los datos.
print("Conjunto de datos cargado correctamente.")
print("Filas y columnas:", df_perfilado.shape)

display(df_perfilado.head())

Conjunto de datos cargado correctamente.
Filas y columnas: (16044, 6)


/var/folders/cn/rlqrdxlx7_s99d2kjb3ztf1c0000gn/T/ipykernel_16488/510940844.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_perfilado = pd.read_sql(consulta_perfilado, conexion)


,pago_id,cliente_id,empleado_id,tienda_id,monto,fecha_pago
0,1,1,1,1,2.99,2005-05-25 11:30:37
1,2,1,1,1,0.99,2005-05-28 10:35:23
2,3,1,1,1,5.99,2005-06-15 00:54:12
3,4,1,2,2,0.99,2005-06-15 18:02:53
4,5,1,2,2,9.99,2005-06-15 21:08:46


In [18]:
# ACTIVIDAD 2 - PUNTO 2.1
# Reconocimiento inicial del conjunto de datos.

# Cantidad de filas y columnas.
print("DIMENSIONES DEL DATAFRAME")
print("Filas:", df_perfilado.shape[0])
print("Columnas:", df_perfilado.shape[1])

# Tipos de datos de cada columna.
print("\nTIPOS DE DATOS")
print(df_perfilado.dtypes)

# Primeras cinco filas.
print("\nPRIMERAS CINCO FILAS")
display(df_perfilado.head())

# Muestra aleatoria reproducible de cinco filas.
print("\nMUESTRA ALEATORIA")
display(df_perfilado.sample(n=5, random_state=42))

# Información general del DataFrame.
print("\nINFORMACIÓN GENERAL")
df_perfilado.info()

# Estadísticas descriptivas con percentiles adicionales.
print("\nESTADÍSTICAS DESCRIPTIVAS")
display(
    df_perfilado.describe(
        include="all",
        percentiles=[0.05, 0.25, 0.50, 0.75, 0.95]
    )
)

DIMENSIONES DEL DATAFRAME
Filas: 16044
Columnas: 6

TIPOS DE DATOS
pago_id                 int64
cliente_id              int64
empleado_id             int64
tienda_id               int64
monto                 float64
fecha_pago     datetime64[us]
dtype: object

PRIMERAS CINCO FILAS


,pago_id,cliente_id,empleado_id,tienda_id,monto,fecha_pago
0,1,1,1,1,2.99,2005-05-25 11:30:37
1,2,1,1,1,0.99,2005-05-28 10:35:23
2,3,1,1,1,5.99,2005-06-15 00:54:12
3,4,1,2,2,0.99,2005-06-15 18:02:53
4,5,1,2,2,9.99,2005-06-15 21:08:46



MUESTRA ALEATORIA


,pago_id,cliente_id,empleado_id,tienda_id,monto,fecha_pago
8248,8251,304,2,2,8.99,2005-07-08 17:07:11
4045,4047,148,1,1,0.99,2005-08-20 22:18:00
4147,4149,152,1,1,4.99,2005-08-19 04:27:11
11927,11931,442,1,1,0.99,2005-07-27 18:41:35
7625,7628,282,1,1,2.99,2005-06-17 12:18:36



INFORMACIÓN GENERAL
<class 'pandas.DataFrame'>
RangeIndex: 16044 entries, 0 to 16043
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   pago_id      16044 non-null  int64         
 1   cliente_id   16044 non-null  int64         
 2   empleado_id  16044 non-null  int64         
 3   tienda_id    16044 non-null  int64         
 4   monto        16044 non-null  float64       
 5   fecha_pago   16044 non-null  datetime64[us]
dtypes: datetime64[us](1), float64(1), int64(4)
memory usage: 752.2 KB

ESTADÍSTICAS DESCRIPTIVAS


,pago_id,cliente_id,empleado_id,tienda_id,monto,fecha_pago
count,16044.000000,16044.000000,16044.000000,16044.000000,16044.000000,16044
mean,8024.483732,297.143169,1.498005,1.498005,4.201356,2005-07-23 08:12:53.217526
min,1.000000,1.000000,1.000000,1.000000,0.000000,2005-05-24 22:53:30
5%,804.150000,29.000000,1.000000,1.000000,0.990000,2005-05-29 18:11:35.100000
25%,4012.750000,148.000000,1.000000,1.000000,2.990000,2005-07-07 00:58:40.500000
50%,8024.500000,296.000000,1.000000,1.000000,3.990000,2005-07-28 16:04:32.500000
75%,12036.250000,446.000000,2.000000,2.000000,4.990000,2005-08-17 21:16:23
95%,15245.850000,569.000000,2.000000,2.000000,8.990000,2005-08-22 23:11:48.500000
max,16049.000000,599.000000,2.000000,2.000000,11.990000,2006-02-14 15:16:03
std,4632.695501,172.453136,0.500012,0.500012,2.362961,NaN


In [19]:
# ACTIVIDAD 2  PUNTO 2.2
# Control de valores faltantes y registros duplicados.

# Calculamos cuántos valores faltan en cada columna
# y qué porcentaje representan.
faltantes = pd.DataFrame({
    "cantidad_faltantes": df_perfilado.isna().sum(),
    "porcentaje_faltantes": (
        df_perfilado.isna().mean() * 100
    ).round(2)
})

print("VALORES FALTANTES POR COLUMNA")
display(faltantes)

# Verificamos duplicados según la clave de negocio.
# Cada pago debe tener un pago_id único.
duplicados = df_perfilado.duplicated(
    subset=["pago_id"]
).sum()

print("\nPAGOS DUPLICADOS")
print("Cantidad:", duplicados)
print("Porcentaje:", round(
    duplicados / len(df_perfilado) * 100, 2
), "%")

VALORES FALTANTES POR COLUMNA


,cantidad_faltantes,porcentaje_faltantes
pago_id,0,0.0
cliente_id,0,0.0
empleado_id,0,0.0
tienda_id,0,0.0
monto,0,0.0
fecha_pago,0,0.0



PAGOS DUPLICADOS
Cantidad: 0
Porcentaje: 0.0 %


### Diagnóstico de calidad – Valores faltantes y duplicados

Se analizaron 16.044 registros correspondientes a operaciones de pago, utilizando `pago_id` como clave de negocio.

**Valores faltantes:** no se detectaron valores nulos en ninguna de las seis columnas analizadas. Todas presentan un 0 % de datos faltantes.

**Duplicados:** no se encontraron identificadores `pago_id` repetidos (0 registros; 0 %).

**Decisión de transformación:** no se requiere imputar valores ni eliminar registros duplicados, ya que el conjunto cumple con ambos controles de calidad.

Se conserva la totalidad de los registros para el análisis posterior.

In [20]:
# ACTIVIDAD 2  PUNTO 2.3
# Detección de valores atípicos mediante el rango intercuartílico (IQR).

# Calculamos los cuartiles del monto de los pagos.
Q1 = df_perfilado["monto"].quantile(0.25)
Q3 = df_perfilado["monto"].quantile(0.75)

# Calculamos el rango intercuartílico.
IQR = Q3 - Q1

# Establecemos los límites para detectar valores atípicos.
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

# Identificamos los pagos que quedan fuera de los límites.
pagos_atipicos = df_perfilado[
    (df_perfilado["monto"] < limite_inferior) |
    (df_perfilado["monto"] > limite_superior)
]

# Mostramos los resultados.
print("ANÁLISIS DE VALORES ATÍPICOS - MÉTODO IQR")
print("Primer cuartil (Q1):", round(Q1, 2))
print("Tercer cuartil (Q3):", round(Q3, 2))
print("Rango intercuartílico (IQR):", round(IQR, 2))
print("Límite inferior:", round(limite_inferior, 2))
print("Límite superior:", round(limite_superior, 2))

print("\nCantidad de pagos atípicos:", len(pagos_atipicos))
print(
    "Porcentaje de pagos atípicos:",
    round(len(pagos_atipicos) / len(df_perfilado) * 100, 2),
    "%"
)

# Revisamos qué montos fueron identificados como atípicos.
print("\nDISTRIBUCIÓN DE MONTOS ATÍPICOS")
display(
    pagos_atipicos["monto"]
    .value_counts()
    .sort_index()
)

ANÁLISIS DE VALORES ATÍPICOS - MÉTODO IQR
Primer cuartil (Q1): 2.99
Tercer cuartil (Q3): 4.99
Rango intercuartílico (IQR): 2.0
Límite inferior: -0.01
Límite superior: 7.99

Cantidad de pagos atípicos: 857
Porcentaje de pagos atípicos: 5.34 %

DISTRIBUCIÓN DE MONTOS ATÍPICOS


monto
8.97       1
8.99     485
9.98       1
9.99     256
10.99    104
11.99     10
Name: count, dtype: int64

### Diagnóstico de calidad – Valores atípicos

Se aplicó el método del rango intercuartílico (IQR) sobre la columna `monto`, correspondiente al importe individual de cada pago.

Se obtuvo un primer cuartil (Q1) de $2,99 y un tercer cuartil (Q3) de $4,99. El rango intercuartílico fue de $2,00, con un límite inferior de -$0,01 y un límite superior de $7,99.

Se identificaron **857 pagos atípicos (5,34 % del total)**, todos superiores al límite máximo calculado.

**Interpretación:** estos valores representan pagos elevados respecto de la distribución general, pero su clasificación estadística no demuestra que sean errores de carga.

**Decisión preliminar:** conservar los 857 registros mientras se verifica su validez mediante controles de negocio. No se recomienda eliminarlos automáticamente, ya que hacerlo podría subestimar los ingresos y alterar el ticket promedio.

In [21]:
# ACTIVIDAD 2  PUNTO 2.4
# Control específico: pagos sin empleado asignado.

# Consultamos la tabla de pagos y verificamos que
# cada empleado_id exista realmente en la tabla empleado.

consulta_pagos_sin_empleado = f"""
SELECT COUNT(*) AS pagos_sin_empleado
FROM {nombre_tabla_sql('payment')} AS p
LEFT JOIN {nombre_tabla_sql('staff')} AS e
    ON p.{nombre_columna_sql('payment', 'staff_id')} = e.{nombre_columna_sql('staff', 'staff_id')}
WHERE e.{nombre_columna_sql('staff', 'staff_id')} IS NULL
"""

# Ejecutamos la consulta en SQL Server.
df_pagos_sin_empleado = pd.read_sql(
    consulta_pagos_sin_empleado,
    conexion
)

print("CONTROL: PAGOS SIN EMPLEADO ASIGNADO")
display(df_pagos_sin_empleado)

CONTROL: PAGOS SIN EMPLEADO ASIGNADO


/var/folders/cn/rlqrdxlx7_s99d2kjb3ztf1c0000gn/T/ipykernel_16488/3258997761.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_pagos_sin_empleado = pd.read_sql(


,pagos_sin_empleado
0,0


In [22]:
# ACTIVIDAD 2 PUNTO 2.4
# Control específico: empleados sin operaciones registradas.

# Partimos de todos los empleados y buscamos sus pagos.
# LEFT JOIN permite incluir también a los empleados
# que no tengan ninguna operación registrada.

consulta_empleados_sin_operaciones = f"""
SELECT
    e.{nombre_columna_sql('staff', 'staff_id')} AS empleado_id,
    e.{nombre_columna_sql('staff', 'first_name')} AS nombre,
    e.{nombre_columna_sql('staff', 'last_name')} AS apellido,
    e.{nombre_columna_sql('staff', 'store_id')} AS tienda_id,
    COUNT(p.{nombre_columna_sql('payment', 'payment_id')}) AS cantidad_pagos
FROM {nombre_tabla_sql('staff')} AS e
LEFT JOIN {nombre_tabla_sql('payment')} AS p
    ON e.{nombre_columna_sql('staff', 'staff_id')} = p.{nombre_columna_sql('payment', 'staff_id')}
GROUP BY
    e.{nombre_columna_sql('staff', 'staff_id')},
    e.{nombre_columna_sql('staff', 'first_name')},
    e.{nombre_columna_sql('staff', 'last_name')},
    e.{nombre_columna_sql('staff', 'store_id')}
HAVING COUNT(p.{nombre_columna_sql('payment', 'payment_id')}) = 0
"""

df_empleados_sin_operaciones = pd.read_sql(
    consulta_empleados_sin_operaciones,
    conexion
)

print("CONTROL: EMPLEADOS SIN OPERACIONES")
print("Cantidad:", len(df_empleados_sin_operaciones))

display(df_empleados_sin_operaciones)

CONTROL: EMPLEADOS SIN OPERACIONES
Cantidad: 0


/var/folders/cn/rlqrdxlx7_s99d2kjb3ztf1c0000gn/T/ipykernel_16488/1459733357.py:26: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_empleados_sin_operaciones = pd.read_sql(


,empleado_id,nombre,apellido,tienda_id,cantidad_pagos


In [23]:
# ACTIVIDAD 2  PUNTO 2.4
# Control específico: clientes con pagos procesados en ambas tiendas.

# Agrupamos las operaciones por cliente y contamos
# en cuántas tiendas diferentes se procesaron sus pagos.

consulta_clientes_dos_tiendas = f"""
SELECT
    p.{nombre_columna_sql('payment', 'customer_id')} AS cliente_id,
    COUNT(DISTINCT e.{nombre_columna_sql('staff', 'store_id')}) AS cantidad_tiendas
FROM {nombre_tabla_sql('payment')} AS p
INNER JOIN {nombre_tabla_sql('staff')} AS e
    ON p.{nombre_columna_sql('payment', 'staff_id')} = e.{nombre_columna_sql('staff', 'staff_id')}
GROUP BY p.{nombre_columna_sql('payment', 'customer_id')}
HAVING COUNT(DISTINCT e.{nombre_columna_sql('staff', 'store_id')}) > 1
"""

df_clientes_dos_tiendas = pd.read_sql(
    consulta_clientes_dos_tiendas,
    conexion
)

print("CONTROL: CLIENTES CON PAGOS EN AMBAS TIENDAS")
print("Cantidad de clientes:", len(df_clientes_dos_tiendas))

display(df_clientes_dos_tiendas.head(10))

CONTROL: CLIENTES CON PAGOS EN AMBAS TIENDAS
Cantidad de clientes: 599


/var/folders/cn/rlqrdxlx7_s99d2kjb3ztf1c0000gn/T/ipykernel_16488/2297248684.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_clientes_dos_tiendas = pd.read_sql(


,cliente_id,cantidad_tiendas
0,1,2
1,2,2
2,3,2
3,4,2
4,5,2
5,6,2
6,7,2
7,8,2
8,9,2
9,10,2


### Diagnóstico de calidad – Controles específicos de la variante 5

Se ejecutaron los tres controles de calidad obligatorios correspondientes al rendimiento comercial por tienda y empleado.

**1. Pagos sin empleado asignado:** se encontraron 0 casos. Todos los pagos tienen un empleado válido asociado.

**2. Empleados sin operaciones registradas:** se encontraron 0 casos. Todos los empleados tienen al menos un pago registrado.

**3. Clientes con operaciones en ambas tiendas:** se identificaron 599 clientes con pagos procesados por empleados de las dos tiendas.

**Interpretación:** los dos primeros controles no presentan inconsistencias. El tercero evidencia que los clientes pueden tener operaciones asociadas a ambas tiendas, aunque esto no demuestra que hayan concurrido físicamente a las dos sucursales.

**Decisión de transformación:** no se requiere eliminar ni corregir registros por estos hallazgos. Se conservarán los datos originales y se tendrá en cuenta la diferencia entre la tienda que procesa el pago y la tienda de origen del cliente al interpretar los resultados comerciales.

In [24]:
# ACTIVIDAD 2  PUNTO 2.5
# Verificación de dos reglas de negocio.

# REGLA 1: los pagos no pueden tener montos negativos.
pagos_negativos = (df_perfilado["monto"] < 0).sum()

# REGLA 2: los pagos registrados no pueden tener
# una fecha posterior al momento del análisis.
fecha_extraccion = pd.Timestamp.now()
pagos_fecha_futura = (
    df_perfilado["fecha_pago"] > fecha_extraccion
).sum()

# Resumimos los resultados.
reglas_negocio = pd.DataFrame({
    "regla": [
        "Pagos con monto negativo",
        "Pagos con fecha futura"
    ],
    "filas_que_incumplen": [
        pagos_negativos,
        pagos_fecha_futura
    ]
})

print("CONTROL DE REGLAS DE NEGOCIO")
display(reglas_negocio)

CONTROL DE REGLAS DE NEGOCIO


,regla,filas_que_incumplen
0,Pagos con monto negativo,0
1,Pagos con fecha futura,0


### Diagnóstico de calidad – Reglas de negocio

Se verificaron dos reglas de negocio sobre los 16.044 registros del conjunto de pagos.

**Regla 1 – Montos no negativos:** se identificaron 0 pagos con importes inferiores a cero. Todos los registros cumplen con la condición establecida.

**Regla 2 – Fechas válidas:** se identificaron 0 pagos con fechas posteriores al momento del análisis. Todos los registros cumplen con esta condición.

**Decisión de transformación:** no se requieren correcciones por incumplimiento de estas reglas. Se conservan los registros originales.

Estos controles complementan la detección estadística de valores atípicos, ya que permiten evaluar condiciones de validez propias del negocio.

In [25]:
# ACTIVIDAD 2  PUNTO 2.6
# Verificación de calidad directamente en SQL Server.
# Controles: montos negativos y pagos sin empleado válido.

consulta_controles_sql = f"""
SELECT
    COUNT(*) AS total_pagos,
    SUM(CASE WHEN p.{nombre_columna_sql('payment', 'amount')} < 0 THEN 1 ELSE 0 END)
        AS pagos_negativos,
    SUM(CASE WHEN e.{nombre_columna_sql('staff', 'staff_id')} IS NULL THEN 1 ELSE 0 END)
        AS pagos_sin_empleado
FROM {nombre_tabla_sql('payment')} AS p
LEFT JOIN {nombre_tabla_sql('staff')} AS e
    ON p.{nombre_columna_sql('payment', 'staff_id')} = e.{nombre_columna_sql('staff', 'staff_id')}
"""

df_controles_sql = pd.read_sql(
    consulta_controles_sql,
    conexion
)

print("CONTROLES DE CALIDAD EN SQL SERVER")
display(df_controles_sql)

CONTROLES DE CALIDAD EN SQL SERVER


/var/folders/cn/rlqrdxlx7_s99d2kjb3ztf1c0000gn/T/ipykernel_16488/2698768672.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_controles_sql = pd.read_sql(


,total_pagos,pagos_negativos,pagos_sin_empleado
0,16044,0,0


In [26]:
# ACTIVIDAD 2  PUNTO 2.6
# Comparación independiente de controles entre SQL Server y pandas.

# Obtenemos los identificadores válidos de empleados.
df_empleados = pd.read_sql(
    f"SELECT {nombre_columna_sql('staff', 'staff_id')} AS empleado_id FROM {nombre_tabla_sql('staff')}",
    conexion
)

# Realizamos ambos controles con pandas.
pagos_negativos_pandas = int(
    (df_perfilado["monto"] < 0).sum()
)

pagos_sin_empleado_pandas = int(
    (
        df_perfilado["empleado_id"].isna()
        | ~df_perfilado["empleado_id"].isin(df_empleados["empleado_id"])
    ).sum()
)

# Comparamos los resultados de pandas con los de SQL Server.
comparacion = pd.DataFrame({
    "control": ["pagos_negativos", "pagos_sin_empleado"],
    "SQL_Server": [
        int(df_controles_sql.iloc[0]["pagos_negativos"]),
        int(df_controles_sql.iloc[0]["pagos_sin_empleado"])
    ],
    "pandas": [
        pagos_negativos_pandas,
        pagos_sin_empleado_pandas
    ]
})

comparacion["coinciden"] = (
    comparacion["SQL_Server"] == comparacion["pandas"]
)

print("COMPARACIÓN DE CONTROLES DE CALIDAD")
display(comparacion)

COMPARACIÓN DE CONTROLES DE CALIDAD


/var/folders/cn/rlqrdxlx7_s99d2kjb3ztf1c0000gn/T/ipykernel_16488/2593162804.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_empleados = pd.read_sql(


,control,SQL_Server,pandas,coinciden
0,pagos_negativos,0,0,True
1,pagos_sin_empleado,0,0,True


### Diagnóstico de calidad – Comparación entre SQL Server y pandas

Se verificaron dos controles de calidad mediante cálculos independientes en T-SQL y pandas.

**Pagos con monto negativo:** ambos métodos identificaron 0 registros.

**Pagos sin empleado válido:** ambos métodos identificaron 0 registros.

Los resultados coincidieron en los dos controles, lo que confirma la consistencia de las verificaciones realizadas sobre el conjunto de datos.

**Decisión de transformación:** no se requieren correcciones para estas condiciones, ya que no se detectaron incumplimientos.

In [27]:
# Consultamos las columnas de pago y alquiler
# para identificar cómo se relacionan las tablas.

for tabla in [nombre_tabla_sql("payment"), nombre_tabla_sql("rental")]:
    print(f"\nColumnas de la tabla {tabla}:")

    consulta = """
    SELECT COLUMN_NAME, DATA_TYPE
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_NAME = ?
    ORDER BY ORDINAL_POSITION
    """

    df_columnas = pd.read_sql(
        consulta,
        conexion,
        params=(tabla,)
    )

    display(df_columnas)


Columnas de la tabla [pago]:


/var/folders/cn/rlqrdxlx7_s99d2kjb3ztf1c0000gn/T/ipykernel_16488/3874785165.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_columnas = pd.read_sql(


,COLUMN_NAME,DATA_TYPE



Columnas de la tabla [alquiler]:


/var/folders/cn/rlqrdxlx7_s99d2kjb3ztf1c0000gn/T/ipykernel_16488/3874785165.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_columnas = pd.read_sql(


,COLUMN_NAME,DATA_TYPE


In [28]:
# Verificamos si los pagos atípicos están asociados
# a un alquiler registrado y si corresponden al mismo cliente.

consulta_validacion_atipicos = f"""
SELECT
    COUNT(*) AS total_atipicos,
    SUM(CASE
        WHEN a.{nombre_columna_sql('rental', 'rental_id')} IS NULL THEN 1 ELSE 0
    END) AS sin_alquiler,
    SUM(CASE
        WHEN a.{nombre_columna_sql('rental', 'rental_id')} IS NOT NULL
             AND p.{nombre_columna_sql('payment', 'customer_id')} <> a.{nombre_columna_sql('rental', 'customer_id')}
        THEN 1 ELSE 0
    END) AS cliente_inconsistente
FROM {nombre_tabla_sql('payment')} AS p
LEFT JOIN {nombre_tabla_sql('rental')} AS a
    ON p.{nombre_columna_sql('payment', 'rental_id')} = a.{nombre_columna_sql('rental', 'rental_id')}
WHERE p.{nombre_columna_sql('payment', 'amount')} > ?
"""

df_validacion_atipicos = pd.read_sql(
    consulta_validacion_atipicos,
    conexion,
    params=(float(limite_superior),)
)

display(df_validacion_atipicos)

/var/folders/cn/rlqrdxlx7_s99d2kjb3ztf1c0000gn/T/ipykernel_16488/246946383.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_validacion_atipicos = pd.read_sql(


,total_atipicos,sin_alquiler,cliente_inconsistente
0,857,0,0
